In [0]:
# Instala os pacotes necessários para o projeto
%pip install python-dotenv azure-storage-file-datalake azure-identity pandas pyarrow pyodbc --quiet

dbutils.library.restartPython()

# Checa se existe um arquivo .env no ambiente
import os
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Variáveis encontradas: {dotenv_path} ")
else:
    print("Variáveis não encontradas.")

# Monta um dicionário reunindo as credenciais de acesso
adls_credentials = {
    "ADLS_CLIENT_ID": os.getenv("ADLS_CLIENT_ID"),
    "ADLS_TENANT_ID": os.getenv("ADLS_TENANT_ID"),
    "ADLS_CLIENT_SECRET": os.getenv("ADLS_CLIENT_SECRET"),
    "STORAGE_ACCOUNT_NAME": os.getenv("STORAGE_ACCOUNT_NAME"),
    "ADLS_CONTAINER": os.getenv("ADLS_CONTAINER", "raw")
}

jdbc_credentials = {
    "SQL_HOST": os.getenv("SQL_HOST"),
    "SQL_DATABASE": os.getenv("SQL_DATABASE"),
    "SQL_USERNAME": os.getenv("SQL_USERNAME"),
    "SQL_PASSWORD": os.getenv("SQL_PASSWORD")
}

print("\nStatus das Credenciais ADLS")

for k, v in adls_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")

print("\nStatus das Credenciais SQL Server")

for k, v in jdbc_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")


all_defined = all(
    list(adls_credentials.values()) +
    list(jdbc_credentials.values())
)

if all_defined:
    print("\nVariáveis carregadas!")
else:
    print("\nExistem variáveis pendentes!")

# Configurações OAuth que serão usadas pelo Spark para autenticação
adls_options = {
    f"fs.azure.account.auth.type.{adls_credentials['STORAGE_ACCOUNT_NAME']}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_credentials['STORAGE_ACCOUNT_NAME']}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_credentials['STORAGE_ACCOUNT_NAME']}.dfs.core.windows.net": adls_credentials["ADLS_CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{adls_credentials['STORAGE_ACCOUNT_NAME']}.dfs.core.windows.net": adls_credentials["ADLS_CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{adls_credentials['STORAGE_ACCOUNT_NAME']}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_credentials['ADLS_TENANT_ID']}/oauth2/token"
}

# Aponta o endereço da pasta batch-data dentro do Container
batch_data_path = (
    f"abfss://{adls_credentials['ADLS_CONTAINER']}@{adls_credentials['STORAGE_ACCOUNT_NAME']}.dfs.core.windows.net/"
    "batch-data"
)

# Busca e exibe os itens presentes no Container
files = (
    spark.read
    .format("binaryFile")
    .options(**adls_options)
    .load(batch_data_path)
    .select("path")
)

files.show(truncate=False)

# LEITURA DE PHYSICAL_VENDAS_CAIXA

# Define o caminho do dataset
physical_vendas_caixa_path = (
    f"abfss://{adls_credentials['ADLS_CONTAINER']}@{adls_credentials['STORAGE_ACCOUNT_NAME']}.dfs.core.windows.net/"
    "batch-data/physical_vendas_caixa.csv"
)

# Leitura do dataset como um Spark DataFrame
df_physical_vendas_caixa_raw = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(physical_vendas_caixa_path)
)

df_physical_vendas_caixa_raw.printSchema()
display(df_physical_vendas_caixa_raw.limit(100000))

# DATA QUALITY: physical_vendas_caixa

from pyspark.sql.functions import col, sum as spark_sum, when, trim, current_timestamp, length

df = df_physical_vendas_caixa_raw

print(f"{'='*60}")
print(f"RELATÓRIO DE QUALIDADE: physical_vendas_caixa")
print(f"{'='*60}")

total = df.count()
print(f"Total de linhas: {total}")

# 1. Nulos por coluna
print("\n--- Valores nulos por coluna ---")
df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

# 2. id_transacao duplicado (deveria ser chave única)
print("--- Duplicidade em id_transacao ---")
dups = df.groupBy("id_transacao").count().filter("count > 1")
print(f"Transações duplicadas: {dups.count()}")
dups.show(truncate=False)

# 3. Linhas totalmente duplicadas
print("--- Linhas 100% duplicadas ---")
distintos = df.distinct().count()
print(f"Duplicadas: {total - distintos}")

# 4. valor_total_venda negativo ou zero (venda sem valor é suspeito)
print("--- Valores de venda inválidos (<= 0) ---")
valores_invalidos = df.filter(col("valor_total_venda") <= 0)
print(f"Vendas com valor <= 0: {valores_invalidos.count()}")
valores_invalidos.show()

# 5. dt_venda no futuro (inconsistência de data)
print("--- Vendas com data no futuro ---")
futuras = df.filter(col("dt_venda") > current_timestamp())
print(f"Vendas com data futura: {futuras.count()}")
futuras.show()

# 6. cpf_cliente com formato inválido (CPF tem 11 dígitos)
print("--- CPFs com tamanho inválido ---")
cpf_invalido = df.filter(
    (length(col("cpf_cliente").cast("string")) != 11) & col("cpf_cliente").isNotNull()
)
print(f"CPFs fora do padrão (11 dígitos): {cpf_invalido.count()}")
cpf_invalido.select("id_transacao", "cpf_cliente").show()

# 7. tipo_pagamento: valores distintos (checar inconsistência de escrita)
print("--- Tipos de pagamento distintos ---")
df.select("tipo_pagamento").distinct().show(truncate=False)

# 8. Cardinalidade das colunas-chave
print("--- Cardinalidade ---")
for c in ["id_loja", "id_caixa", "id_operador", "tipo_pagamento"]:
    distintos_c = df.select(c).distinct().count()
    print(f"  {c}: {distintos_c} valores únicos")


